In [48]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2026, 1, 12, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 1, 12, 23, 59))

# from SDRUtils.data.builder import SDRDataBuilder
# sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
# df = sdr.grab_sdr_trades(
# 	start_timestamp=start,
# 	end_timestamp=end,
# 	agency="CFTC",
# 	asset_class="RATES",
# )
# df

In [50]:
# from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
# USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path)

from SDRUtils.products.usd.usd_swaptions import USD_Swaptions
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)
# sdf

Classifying Trades: 100%|██████████| 646/646 [00:00<00:00, 1346.85trade/s]


here
None USD-SOFR-COMPOUND 1D CONSTANT 2Yx10Y PAYER EURO VANILLA PHYS / USD-SOFR-COMPOUND 1D CONSTANT 2Yx10Y RECEIVER EURO VANILLA PHYS 492
None USD-SOFR-OIS Compound 1D CONSTANT 1Mx10Y PAYER EURO VANILLA PHYS / USD-SOFR-OIS Compound 1D CONSTANT 1Mx10Y RECEIVER EURO VANILLA PHYS 493
15245.492855900586 USD-SOFR-OIS Compound 1D CONSTANT 1Mx10Y PAYER EURO VANILLA PHYS / USD-SOFR-OIS Compound 1D CONSTANT 1Mx10Y RECEIVER EURO VANILLA PHYS 494
None USD-SOFR-OIS Compound 3M CONSTANT 9Mx22Y PAYER EURO VANILLA PHYS / USD-SOFR-OIS Compound 3M CONSTANT 9Mx22Y RECEIVER EURO VANILLA PHYS 495
1949.9909743463 USD-SOFR-OIS Compound 1D CONSTANT 1Mx10Y PAYER EURO VANILLA PHYS / USD-SOFR-OIS Compound 1D CONSTANT 1Mx10Y RECEIVER EURO VANILLA PHYS 497
36057.5151007636 USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y PAYER EURO VANILLA PHYS / USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y RECEIVER EURO VANILLA PHYS 498
25755.367929116852 USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y PAYER EURO VANILLA PHYS / USD-SOFR-OIS Com

ArrowTypeError: ("object of type <class 'str'> cannot be converted to int", 'Conversion failed for column execution_timestamp with type object')

In [9]:
# temp = sdf.copy()
# temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
# temp[temp["package_type"] == "SWAPTION"].to_excel("filter_swaptions_trades_no_pkg.xlsx")
# sdf["trade_label"].value_counts()


# sdf["package_type"].value_counts()
# sdf[~sdf["trade_label"].str.contains("4Dx10Y")]["package_type"].value_counts()
# sdf[(sdf["package_type"] == "STRADDLE") & (sdf["trade_label"].str.contains("1Mx2Y"))]
# sdf[(sdf["package_type"] == "STRADDLE") & (sdf["trade_label"].str.contains("1Yx10Y"))]
# sdf[sdf["trade_label"].str.contains("3Mx10Y")]

In [23]:
sdf[sdf["package_id"] == "STRADDLE_c21f66b0f51c"]

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count,vega_curve_type,vega_curve_id,vega_curve_legs,vega_curve_vega01,vega_curve_pricing_method
513,NEWT-TRAD / MODI-TRAD,1696911254000000101 / 1696922111000000101,2026-01-12 15:55:38+00:00,2026-01-12,2031-01-13,SWAPTION_PAYER / SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 5Yx30Y PAYER...,100000000.0,USD,False,...,"21,970,000","21,970,000",1.0,platform=ISWV; time_delta_max=0.0s; premium_mo...,2.0,NaN,NaN,NaN,NaN,NaN


In [34]:
# ["platform_identifier"].value_counts()
# sdf[sdf["package_type"] == "RISK_REVERSAL"]
# sdf = sdf[~(sdf["trade_label"].str.contains("4Dx10Y")) & (sdf["package_type"] == "SWAPTION") & ~(sdf["platform_identifier"].isin(["BILT", "XXXX"]))]
# sdf["execution_timestamp"] = sdf["execution_timestamp"].astype(str)
# sdf.to_excel("temp1.xlsx")

# sdf[(sdf["package_type"] == "STRADDLE") & (sdf[ "package_indicator"] == False)]
# sdf["package_type"].value_counts()
# sdf[(sdf["package_type"] == "VERTICAL_SPREAD_1x1")]

In [51]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))
pricer

QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x0000026CF2A68AB0> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x0000026CF2A6A940> >, _meta_data={'timestamp': datetime.datetime(2026, 1, 12, 0, 0)})

In [55]:
from SDRUtils.products._swaptions.pricer import usd_swaption_straddle_pricer_from_row, usd_swaption_leg_pricer_from_row

# sdf.loc[285], sdf.loc[286]
# .iloc[0].to_dict()
# df[df["Original Dissemination Identifier"] == 1653435994000001301]
# usd_swaption_leg_pricer_from_row(sdf.loc[491], pricer, leg="payer", fwd_prem=248750)
# usd_swaption_leg_pricer_from_row(sdf.loc[15], pricer, leg="receiver", fwd_prem=565000.0)
usd_swaption_straddle_pricer_from_row(sdf.loc[492], pricer)

RuntimeError: root not bracketed: f[0,1] -> [1.333380e+09,1.391770e+09]

In [56]:
# sdf.loc[336]["package_reason"]
sdf.loc[492]

event_action                                                          NEWT-TRAD
trade_id                              1693390051000000201 / 1693444086000002901
execution_timestamp                                   2026-01-12 06:46:25+00:00
effective_date                                              2026-01-12 00:00:00
expiration_date                                             2028-01-12 00:00:00
product_type                                 SWAPTION_PAYER / SWAPTION_RECEIVER
trade_label                   USD-SOFR-COMPOUND 1D CONSTANT 2Yx10Y PAYER EUR...
notional                                                            100000000.0
notional_currency                                                           CNY
is_notional_capped                                                        False
estimated_pv01                                                              0.0
package_type                                                           STRADDLE
package_id                              

In [46]:
4.09* np.sqrt(252)

np.float64(64.92673717352505)

In [51]:
ids = [
1696946219000001501,
1696946220000001601,
1696946221000001701,
1696949372000000201,


]

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_csv("temp_trades.csv",index=False)

sdf[sdf["trade_id"].isin([str(id) for id in ids])].to_dict(orient="records")

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df["Action type"].value_counts()
# df["Event type"].value_counts()

[{'event_action': 'NEWT-TRAD',
  'trade_id': '1696946220000001601',
  'execution_timestamp': Timestamp('2026-01-12 16:10:46+0000', tz='UTC'),
  'effective_date': Timestamp('2026-01-12 00:00:00'),
  'expiration_date': Timestamp('2026-02-12 00:00:00'),
  'product_type': 'SWAPTION_RECEIVER',
  'trade_label': 'USD-SOFR-COMPOUND 1D CONSTANT 1Mx20Y RECEIVER EURO VANILLA PHYS',
  'notional': 25000000.0,
  'notional_currency': 'USD',
  'is_notional_capped': False,
  'estimated_pv01': 0.0,
  'package_type': 'SWAPTION',
  'package_id': None,
  'package_legs': None,
  'underlying_expiration_date': Timestamp('2046-02-17 00:00:00'),
  'tenor_years': 20.027397260273972,
  'tenor_label': '20Y',
  'forward_start_years': 0.08493150684931507,
  'forward_label': '1M',
  'premium': 248750.0,
  'exercise_style': 'EUROPEAN',
  'strike': 0.0417,
  'upi_underlier_name': 'NA/Swap Fxd Flt USD',
  'unique_product_identifier': 'QZXZSN00ZVCG',
  'platform_identifier': 'BILT',
  'cleared': 'N',
  'package_indicator